# Memory Bandit Reinstatement Analysis

This notebook performs a reinstatement analysis using MVPA-style classification on Morlet TFRs.
We train a classifier on 'feedback_start' periods to predict RPE category (negative, neutral, positive) and test it on 'cue_on' periods. We of course can only do this for items that were successfully retrieved. 

We should maybe then target those encoding level features and look at what they do during non-encoded items.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from mne.decoding import GeneralizingEstimator
import warnings
from tqdm import tqdm
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)


In [ ]:
# Import plot configuration
from plot_config import set_plot_style
set_plot_style()


## Data Loading and Setup


In [ ]:
# Set up paths
base_dir = '/Volumes/T7 Shield'  # Root directory for data
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load demographics data
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Filter out specific subjects
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subj_ids = subj_df.MSSMCode.values
print(f"Available subjects: {subj_ids}")


In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)


# # First, let's drop reaction times that are physiologically too fast
# learn_df = learn_df[learn_df.gamble_rt>0.3]
# drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
#                           combined_df[combined_df.recog_rt<0.3]])
# combined_df.drop(index=drop_trials.index, inplace=True)

# # Next, let's drop reaction times that are slower than the 97.5th percentile of each subject's reaction times
# for subj in learn_df.participant.unique():
#     rt_threshold = np.percentile(learn_df[learn_df.participant==subj].gamble_rt, 95)
#     learn_df.loc[(learn_df.participant==subj) & (learn_df.gamble_rt>rt_threshold), 'gamble_rt'] = np.nan
#     combined_df.loc[(combined_df.participant==subj) & (combined_df.gamble_rt_gamble>rt_threshold), 'gamble_rt_gamble'] = np.nan
#     learn_df.loc[(learn_df.participant==subj), 'rpe_cat'] = pd.qcut(learn_df.loc[(learn_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])
#     combined_df.loc[(combined_df.participant==subj), 'rpe_cat'] = pd.qcut(combined_df.loc[(combined_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])



learn_df.dropna(subset=['gamble_rt'], inplace=True)
combined_df.dropna(subset=['gamble_rt_gamble'], inplace=True)

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((combined_df.loc[combined_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        combined_df.loc[combined_df.participant==subj,
                'RT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'reward_gamble'] - RT_inferred_EV_subj)

        combined_df.loc[combined_df.participant==subj,
                'zRT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'RT_rpe']))

## Select Subject and Load TFR Data


In [ ]:
# Multi-subject analysis parameters
day = 'Day1'
events = ['feedback_start', 'cue_on']
target_roi = 'OFC'  # Target ROI for analysis: 'dmPFC', 'HPC', 'AMY', 'ACC', 'OFC'

# TFR parameters
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

# Analysis parameters
min_electrodes_per_roi = 1  # Minimum electrodes required for ROI analysis
hfa_freq_range = (70, 200)  # HFA frequency range
sliding_window_ms = 200  # Sliding window for smoothing

# Results storage
results_dir = f'{base_dir}/scratch/MemoryBandit/reinstatement_results'
os.makedirs(results_dir, exist_ok=True)
roi_results_dir = f'{results_dir}/{target_roi}'
os.makedirs(roi_results_dir, exist_ok=True)

print(f"Multi-subject reinstatement analysis setup:")
print(f"  Target ROI: {target_roi}")
print(f"  Subjects to analyze: {len(subj_ids)}")
print(f"  Frequency range: {freqs[0]:.1f} - {freqs[-1]:.1f} Hz")
print(f"  HFA range: {hfa_freq_range[0]}-{hfa_freq_range[1]} Hz")
print(f"  Results directory: {roi_results_dir}")
print(f"  Min electrodes per ROI: {min_electrodes_per_roi}")


In [ ]:
# ROI validation and electrode selection functions
def validate_roi_electrodes(subj_id, target_roi, min_electrodes=1):
    """
    Validate that a subject has enough electrodes in the target ROI
    """
    try:
        elec_file = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df'
        if not os.path.exists(elec_file):
            return False, 0, None
        
        elec_df = pd.read_csv(elec_file)
        roi_elecs = elec_df[elec_df.salman_region == target_roi]
        
        if len(roi_elecs) >= min_electrodes:
            return True, len(roi_elecs), roi_elecs
        else:
            return False, len(roi_elecs), roi_elecs
            
    except Exception as e:
        print(f"Error validating ROI for {subj_id}: {e}")
        return False, 0, None

def load_subject_tfr_data(subj_id, events):
    """
    Load TFR data for a subject
    """
    tfr_data = {}
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    
    for event in events:
        try:
            tfr_file = f'{filepath}/{event}-tfr.h5'
            if os.path.exists(tfr_file):
                tfr_data[event] = mne.time_frequency.read_tfrs(tfr_file)
            else:
                print(f"TFR file not found for {subj_id}: {tfr_file}")
                tfr_data[event] = None
        except Exception as e:
            print(f"Error loading TFR for {subj_id} {event}: {e}")
            tfr_data[event] = None
    
    return tfr_data

# Test ROI validation for all subjects
print("Validating ROI availability across subjects:")
valid_subjects = []
roi_counts = {}

for subj_id in subj_ids:
    is_valid, n_elecs, elec_df = validate_roi_electrodes(subj_id, target_roi, min_electrodes_per_roi)
    roi_counts[subj_id] = n_elecs
    
    if is_valid:
        valid_subjects.append(subj_id)
        print(f"  {subj_id}: ✓ {n_elecs} electrodes")
    else:
        print(f"  {subj_id}: ✗ {n_elecs} electrodes (insufficient)")

print(f"\nValid subjects for {target_roi} analysis: {len(valid_subjects)}/{len(subj_ids)}")
print(f"Valid subjects: {valid_subjects}")


In [ ]:
# Main analysis function for single subject
def run_reinstatement_analysis(subj_id, target_roi, events, learn_df, 
                              hfa_freq_range, sliding_window_ms, 
                              min_electrodes_per_roi, run_cluster_analysis=True):
    """
    Run reinstatement analysis for a single subject
    """
    print(f"\n=== Processing {subj_id} ===")
    
    # Validate ROI
    is_valid, n_elecs, roi_elecs = validate_roi_electrodes(subj_id, target_roi, min_electrodes_per_roi)
    if not is_valid:
        print(f"Skipping {subj_id}: insufficient electrodes ({n_elecs})")
        return None
    
    # Load TFR data
    tfr_data = load_subject_tfr_data(subj_id, events)
    if any(tfr_data[event] is None for event in events):
        print(f"Skipping {subj_id}: missing TFR data")
        return None
    
    # Get subject behavioral data
    subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
    if len(subj_learn_df) == 0:
        print(f"Skipping {subj_id}: no behavioral data")
        return None
    
    # Ensure we have the right number of trials
    n_trials = tfr_data['feedback_start'].data.shape[0]
    if len(subj_learn_df) != n_trials:
        print(f"Warning: {subj_id} has {len(subj_learn_df)} behavioral trials, but {n_trials} TFR trials")
        subj_learn_df = subj_learn_df.head(n_trials)
    
    # Create RPE categories
    rpe_values = subj_learn_df['rpe'].values
    rpe_cat = []
    for rpe in rpe_values:
        if rpe < -0.1:
            rpe_cat.append('neg')
        elif rpe > 0.1:
            rpe_cat.append('pos')
        else:
            rpe_cat.append('neutral')
    rpe_cat = np.array(rpe_cat)
    
    # Check class distribution
    unique_cats, counts = np.unique(rpe_cat, return_counts=True)
    min_trials_per_class = min(counts)
    if min_trials_per_class < 5:
        print(f"Skipping {subj_id}: insufficient trials per class (min: {min_trials_per_class})")
        return None
    
    print(f"  RPE distribution: {dict(zip(unique_cats, counts))}")
    
    # Extract ROI channels
    roi_channels = roi_elecs.values
    tfr_roi = {}
    
    for event in events:
        # Find indices of channels in ROI
        ch_indices = [i for i, ch in enumerate(tfr_data[event].ch_names) if ch in roi_channels]
        if not ch_indices:
            print(f"Skipping {subj_id}: no ROI channels found in {event}")
            return None
        
        # Pick only ROI channels
        tfr_roi[event] = tfr_data[event].copy().pick_channels([tfr_data[event].ch_names[i] for i in ch_indices])
        print(f"  {event}: {len(ch_indices)} ROI channels")
    
    # Get data dimensions
    feedback_data = tfr_roi['feedback_start'].data
    cue_data = tfr_roi['cue_on'].data
    n_trials, n_channels, n_freqs, n_times_feedback = feedback_data.shape
    _, _, _, n_times_cue = cue_data.shape
    
    # Define HFA frequency band
    freqs = tfr_roi['feedback_start'].freqs
    hfa_mask = (freqs >= hfa_freq_range[0]) & (freqs <= hfa_freq_range[1])
    hfa_indices = np.where(hfa_mask)[0]
    
    if len(hfa_indices) == 0:
        print(f"Skipping {subj_id}: no HFA frequencies found")
        return None
    
    print(f"  HFA frequencies: {len(hfa_indices)} out of {n_freqs} total")
    
    # Average power across HFA frequencies
    feedback_hfa = np.mean(feedback_data[:, :, hfa_mask, :], axis=2)
    cue_hfa = np.mean(cue_data[:, :, hfa_mask, :], axis=2)
    
    # Apply sliding window smoothing
    def apply_sliding_window(data, window_size_ms=200):
        times = tfr_roi['feedback_start'].times
        sfreq = 1 / (times[1] - times[0])
        window_size_samples = int(window_size_ms * sfreq / 1000)
        window_size_samples = max(1, window_size_samples)
        
        smoothed_data = np.zeros_like(data)
        for trial in range(data.shape[0]):
            for channel in range(data.shape[1]):
                for t in range(data.shape[2]):
                    start_idx = max(0, t - window_size_samples // 2)
                    end_idx = min(data.shape[2], t + window_size_samples // 2 + 1)
                    smoothed_data[trial, channel, t] = np.mean(data[trial, channel, start_idx:end_idx])
        return smoothed_data
    
    feedback_hfa_smooth = apply_sliding_window(feedback_hfa, sliding_window_ms)
    cue_hfa_smooth = apply_sliding_window(cue_hfa, sliding_window_ms)
    
    # Set up classifier
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
    
    # Initialize results matrix
    classification_scores = np.zeros((n_times_feedback, n_times_cue))
    
    print(f"  Running classification for {n_times_feedback} x {n_times_cue} combinations...")
    
    # Classification function
    def classify_timepoint_combination(args):
        fb_time_idx, cue_time_idx = args
        
        X_train = feedback_hfa_smooth[:, :, fb_time_idx]
        X_test = cue_hfa_smooth[:, :, cue_time_idx]
        
        try:
            clf.fit(X_train, rpe_cat)
            y_pred_proba = clf.predict_proba(X_test)
            
            # Calculate macro-averaged AUC
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            y_encoded = le.fit_transform(rpe_cat)
            
            auc_scores = []
            for i in range(len(le.classes_)):
                y_binary = (y_encoded == i).astype(int)
                if len(np.unique(y_binary)) > 1:
                    auc = roc_auc_score(y_binary, y_pred_proba[:, i])
                    auc_scores.append(auc)
            
            if len(auc_scores) > 0:
                return np.mean(auc_scores)
            else:
                return 0.5
                
        except ValueError:
            return 0.5
    
    # Create all combinations
    combinations = [(fb_time_idx, cue_time_idx) 
                   for fb_time_idx in range(n_times_feedback) 
                   for cue_time_idx in range(n_times_cue)]
    
    # Run parallel classification
    from joblib import Parallel, delayed
    n_jobs = min(8, os.cpu_count())
    
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(classify_timepoint_combination)(combo) for combo in combinations
    )
    
    # Reshape results
    result_idx = 0
    for fb_time_idx in range(n_times_feedback):
        for cue_time_idx in range(n_times_cue):
            classification_scores[fb_time_idx, cue_time_idx] = results[result_idx]
            result_idx += 1
    
    print(f"  Classification completed! Score range: {classification_scores.min():.3f} - {classification_scores.max():.3f}")
    
    # Run cluster permutation test if requested
    cluster_results = None
    if run_cluster_analysis:
        print(f"  Running MNE cluster permutation test...")
        try:
            cluster_results = cluster_permutation_test_mne(
                classification_scores, 
                tfr_roi['feedback_start'].times,
                tfr_roi['cue_on'].times,
                n_permutations=1000,
                cluster_threshold=0.05,
                min_cluster_size=5
            )
            print(f"  MNE cluster analysis completed!")
        except Exception as e:
            print(f"  MNE cluster analysis failed: {e}")
            cluster_results = None
    
    # Return results with metadata
    results_dict = {
        'subj_id': subj_id,
        'classification_scores': classification_scores,
        'cluster_results': cluster_results,
        'feedback_times': tfr_roi['feedback_start'].times,
        'cue_times': tfr_roi['cue_on'].times,
        'n_electrodes': n_channels,
        'n_trials': n_trials,
        'rpe_distribution': dict(zip(unique_cats, counts)),
        'hfa_freqs': freqs[hfa_indices]
    }
    
    return results_dict


In [ ]:
# # Cluster permutation test using MNE's correct implementation
# import mne.stats

# def cluster_permutation_test_mne(classification_scores, feedback_times, cue_times, 
#                                 n_permutations=1000, cluster_threshold=0.05, 
#                                 min_cluster_size=5, adjacency=None):
#     """
#     Perform cluster permutation test using MNE's permutation_cluster_1samp_test
    
#     Parameters:
#     -----------
#     classification_scores : array-like
#         Classification scores matrix (feedback_times x cue_times)
#     feedback_times : array-like
#         Time vector for feedback dimension
#     cue_times : array-like
#         Time vector for cue dimension
#     n_permutations : int
#         Number of permutations for cluster test
#     cluster_threshold : float
#         Threshold for cluster formation (p-value)
#     min_cluster_size : int
#         Minimum cluster size in timepoints
#     adjacency : array-like, optional
#         Adjacency matrix for cluster formation
    
#     Returns:
#     --------
#     cluster_results : dict
#         Dictionary containing cluster statistics and significance
#     """
    
#     print(f"Running MNE permutation_cluster_1samp_test...")
#     print(f"  Permutations: {n_permutations}")
#     print(f"  Cluster threshold: {cluster_threshold}")
#     print(f"  Min cluster size: {min_cluster_size}")
    
#     # For single subject, we'll create surrogate data by shuffling trials
#     # In group analysis, this would be the actual subject data
#     n_feedback, n_cue = classification_scores.shape
    
#     # Create adjacency matrix for 2D time grid
#     if adjacency is None:
#         # Create adjacency for 2D time grid (8-connected)
#         adjacency = mne.stats.combine_adjacency(n_feedback, n_cue)
    
#     # For single subject analysis, we'll use the classification scores directly
#     # and test against chance (0.5)
#     data_2d = classification_scores.reshape(1, n_feedback * n_cue)
    
#     # Perform one-sample t-test against chance (0.5)
#     from scipy import stats
#     t_stats = np.zeros(n_feedback * n_cue)
#     p_values = np.zeros(n_feedback * n_cue)
    
#     for i in range(n_feedback * n_cue):
#         t_stat, p_val = stats.ttest_1samp([data_2d[0, i]], 0.5)
#         t_stats[i] = t_stat
#         p_values[i] = p_val
    
#     # Reshape back to 2D
#     t_stats_2d = t_stats.reshape(n_feedback, n_cue)
#     p_values_2d = p_values.reshape(n_feedback, n_cue)
    
#     # Run MNE permutation_cluster_1samp_test
#     try:
#         # For single subject, we need to create a small dataset
#         # We'll use the classification scores as "data" and test against chance
#         data_for_cluster = classification_scores.reshape(1, n_feedback * n_cue)
        
#         # Convert threshold from p-value to t-statistic
#         from scipy.stats import t
#         df = 1 - 1  # degrees of freedom for single observation
#         t_threshold = t.ppf(1 - cluster_threshold / 2, df)  # two-tailed
        
#         # Run permutation cluster test
#         t_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(
#             data_for_cluster, 
#             adjacency=adjacency,
#             n_permutations=n_permutations,
#             threshold=t_threshold,
#             tail=0,  # Two-tailed test
#             out_type='indices'
#         )
        
#         print(f"  MNE permutation_cluster_1samp_test completed")
#         print(f"  Found {len(clusters)} clusters")
#         print(f"  Significant clusters: {np.sum(cluster_p_values < 0.05)}")
        
#         # Create results dictionary
#         cluster_results = {
#             't_obs': t_obs,
#             'clusters': clusters,
#             'cluster_p_values': cluster_p_values,
#             'H0': H0,
#             't_stats': t_stats_2d,
#             'p_values': p_values_2d,
#             'n_permutations': n_permutations,
#             'cluster_threshold': cluster_threshold,
#             'min_cluster_size': min_cluster_size,
#             'adjacency': adjacency
#         }
        
#         return cluster_results
        
#     except Exception as e:
#         print(f"  MNE permutation_cluster_1samp_test failed: {e}")
#         print("  Falling back to simple threshold-based clustering...")
        
#         # Fallback: simple threshold-based clustering
#         sig_mask = p_values_2d < cluster_threshold
#         clusters = find_clusters_2d_simple(sig_mask, min_cluster_size)
        
#         cluster_results = {
#             't_obs': t_stats_2d,
#             'clusters': clusters,
#             'cluster_p_values': [0.5] * len(clusters),  # Placeholder
#             'H0': None,
#             't_stats': t_stats_2d,
#             'p_values': p_values_2d,
#             'n_permutations': n_permutations,
#             'cluster_threshold': cluster_threshold,
#             'min_cluster_size': min_cluster_size,
#             'adjacency': adjacency
#         }
        
#         return cluster_results

# def find_clusters_2d_simple(sig_mask, min_cluster_size):
#     """
#     Simple 2D cluster finding (fallback method)
#     """
#     from scipy.ndimage import label
    
#     # Label connected components
#     labeled_array, num_features = label(sig_mask)
    
#     clusters = []
#     for i in range(1, num_features + 1):
#         cluster_mask = labeled_array == i
#         cluster_coords = np.where(cluster_mask)
        
#         if len(cluster_coords[0]) >= min_cluster_size:
#             clusters.append(cluster_coords)
    
#     return clusters

# def plot_cluster_heatmap_mne(classification_scores, cluster_results, feedback_times, cue_times, 
#                             target_roi, significance_threshold=0.05):
#     """
#     Create heatmap with MNE cluster-based significance overlay
#     """
    
#     fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
#     # Original classification scores
#     ax1 = axes[0, 0]
#     im1 = ax1.imshow(classification_scores, aspect='auto', origin='lower', 
#                      cmap='RdBu_r', vmin=0.3, vmax=0.7,
#                      extent=[cue_times[0], cue_times[-1], 
#                             feedback_times[0], feedback_times[-1]])
#     ax1.set_title(f'{target_roi} Reinstatement - Classification Scores')
#     ax1.set_xlabel('Cue On Time (s)')
#     ax1.set_ylabel('Feedback Start Time (s)')
#     ax1.plot([cue_times[0], cue_times[-1]], [feedback_times[0], feedback_times[-1]], 
#              'k--', alpha=0.5, linewidth=1)
#     plt.colorbar(im1, ax=ax1, label='AUC Score')
    
#     # T-statistics
#     ax2 = axes[0, 1]
#     t_stats = cluster_results['t_stats']
#     im2 = ax2.imshow(t_stats, aspect='auto', origin='lower', 
#                      cmap='RdBu_r', vmin=-3, vmax=3,
#                      extent=[cue_times[0], cue_times[-1], 
#                             feedback_times[0], feedback_times[-1]])
#     ax2.set_title(f'{target_roi} Reinstatement - T-Statistics')
#     ax2.set_xlabel('Cue On Time (s)')
#     ax2.set_ylabel('Feedback Start Time (s)')
#     plt.colorbar(im2, ax=ax2, label='T-statistic')
    
#     # Cluster-based significance
#     ax3 = axes[1, 0]
    
#     # Create cluster mask using MNE results
#     cluster_mask = np.zeros_like(classification_scores, dtype=bool)
#     significant_clusters = []
    
#     if 'clusters' in cluster_results and 'cluster_p_values' in cluster_results:
#         clusters = cluster_results['clusters']
#         cluster_p_values = cluster_results['cluster_p_values']
        
#         # Find significant clusters
#         for i, (cluster, p_val) in enumerate(zip(clusters, cluster_p_values)):
#             if p_val < significance_threshold:
#                 significant_clusters.append((i, cluster, p_val))
                
#                 # Mark cluster locations
#                 if isinstance(cluster, tuple) and len(cluster) == 2:
#                     # Cluster is in indices format
#                     cluster_mask[cluster] = True
#                 elif isinstance(cluster, np.ndarray):
#                     # Cluster is in mask format
#                     cluster_mask |= cluster
    
#     # Plot original scores with significant clusters highlighted
#     im3 = ax3.imshow(classification_scores, aspect='auto', origin='lower', 
#                      cmap='RdBu_r', vmin=0.3, vmax=0.7,
#                      extent=[cue_times[0], cue_times[-1], 
#                             feedback_times[0], feedback_times[-1]])
    
#     # Overlay significant clusters
#     if np.any(cluster_mask):
#         # Plot significant clusters with high contrast
#         ax3.imshow(cluster_mask, aspect='auto', origin='lower', 
#                    cmap='Reds', alpha=0.7, vmin=0, vmax=1,
#                    extent=[cue_times[0], cue_times[-1], 
#                           feedback_times[0], feedback_times[-1]])
    
#     ax3.set_title(f'{target_roi} Reinstatement - Significant Clusters (p<{significance_threshold})')
#     ax3.set_xlabel('Cue On Time (s)')
#     ax3.set_ylabel('Feedback Start Time (s)')
#     plt.colorbar(im3, ax=ax3, label='AUC Score')
    
#     # Cluster statistics summary
#     ax4 = axes[1, 1]
#     ax4.axis('off')
    
#     # Create text summary
#     summary_text = f"MNE Cluster Analysis Summary\\n\\n"
#     summary_text += f"Total clusters found: {len(cluster_results.get('clusters', []))}\\n"
#     summary_text += f"Significant clusters: {len(significant_clusters)}\\n"
#     summary_text += f"Cluster threshold: {cluster_results['cluster_threshold']}\\n"
#     summary_text += f"Min cluster size: {cluster_results['min_cluster_size']}\\n"
#     summary_text += f"Permutations: {cluster_results['n_permutations']}\\n\\n"
    
#     if len(significant_clusters) > 0:
#         summary_text += "Significant Clusters:\\n"
#         for i, (cluster_idx, cluster, p_val) in enumerate(significant_clusters):
#             if isinstance(cluster, tuple) and len(cluster) == 2:
#                 cluster_size = len(cluster[0])
#             else:
#                 cluster_size = np.sum(cluster)
#             summary_text += f"  Cluster {i+1}: p={p_val:.4f}, size={cluster_size}\\n"
    
#     ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, 
#              fontsize=10, verticalalignment='top', fontfamily='monospace')
    
#     plt.tight_layout()
#     plt.show()
    
#     return fig

# # Test the MNE cluster analysis on a single subject (for demonstration)
# if 'tfr_roi' in locals() and tfr_roi['feedback_start'] is not None:
#     print("Testing MNE permutation_cluster_1samp_test on single subject...")
    
#     # Use the existing classification_scores from the previous analysis
#     if 'classification_scores' in locals():
#         # Run MNE cluster permutation test
#         cluster_results = cluster_permutation_test_mne(
#             classification_scores, 
#             tfr_roi['feedback_start'].times,
#             tfr_roi['cue_on'].times,
#             n_permutations=1000,
#             cluster_threshold=0.05,
#             min_cluster_size=5
#         )
        
#         # Create cluster-based heatmap
#         fig = plot_cluster_heatmap_mne(
#             classification_scores, 
#             cluster_results, 
#             tfr_roi['feedback_start'].times, 
#             tfr_roi['cue_on'].times, 
#             target_roi
#         )
        
#         print("MNE cluster analysis completed!")
#     else:
#         print("No classification_scores found. Run the main analysis first.")
# else:
#     print("No TFR data available for cluster testing.")


In [ ]:
# Multi-subject analysis loop
print(f"\n{'='*60}")
print(f"STARTING MULTI-SUBJECT REINSTATEMENT ANALYSIS")
print(f"Target ROI: {target_roi}")
print(f"Valid subjects: {len(valid_subjects)}")
print(f"{'='*60}")

# Storage for all results
all_results = []
successful_subjects = []
failed_subjects = []

# Process each valid subject
for i, subj_id in enumerate(valid_subjects):
    print(f"\nProgress: {i+1}/{len(valid_subjects)} subjects")
    
    try:
        # Run analysis for this subject
        result = run_reinstatement_analysis(
            subj_id=subj_id,
            target_roi=target_roi,
            events=events,
            learn_df=learn_df,
            hfa_freq_range=hfa_freq_range,
            sliding_window_ms=sliding_window_ms,
            min_electrodes_per_roi=min_electrodes_per_roi
        )
        
        if result is not None:
            all_results.append(result)
            successful_subjects.append(subj_id)
            
            # Save individual subject results
            result_file = f'{roi_results_dir}/{subj_id}_reinstatement_results.npy'
            np.save(result_file, result)
            print(f"  Results saved to: {result_file}")
        else:
            failed_subjects.append(subj_id)
            
    except Exception as e:
        print(f"  ERROR processing {subj_id}: {e}")
        failed_subjects.append(subj_id)

print(f"\n{'='*60}")
print(f"ANALYSIS COMPLETED")
print(f"Successful subjects: {len(successful_subjects)}")
print(f"Failed subjects: {len(failed_subjects)}")
print(f"Successful: {successful_subjects}")
if failed_subjects:
    print(f"Failed: {failed_subjects}")
print(f"{'='*60}")

# Save summary metadata
summary_metadata = {
    'target_roi': target_roi,
    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'successful_subjects': successful_subjects,
    'failed_subjects': failed_subjects,
    'analysis_parameters': {
        'hfa_freq_range': hfa_freq_range,
        'sliding_window_ms': sliding_window_ms,
        'min_electrodes_per_roi': min_electrodes_per_roi,
        'events': events
    },
    'roi_counts': roi_counts
}

summary_file = f'{roi_results_dir}/analysis_summary.json'
import json
with open(summary_file, 'w') as f:
    json.dump(summary_metadata, f, indent=2)

print(f"Summary metadata saved to: {summary_file}")


In [ ]:
all_results =[]
# Load individual subject results
for subj_id in valid_subjects:
    result_file = f'{roi_results_dir}/{subj_id}_reinstatement_results.npy'
    result = np.load(result_file, allow_pickle=True)
    all_results.append(result.item())


In [ ]:
# Aggregation and visualization functions
def aggregate_subject_results(all_results):
    """
    Aggregate results across subjects
    """
    if len(all_results) == 0:
        print("No results to aggregate!")
        return None
    
    # Get common dimensions
    n_subjects = len(all_results)
    n_feedback_times, n_cue_times = all_results[0]['classification_scores'].shape
    
    # Stack all classification scores
    all_scores = np.array([result['classification_scores'] for result in all_results])
    
    # Calculate group statistics
    group_mean = np.mean(all_scores, axis=0)
    group_std = np.std(all_scores, axis=0)
    group_sem = group_std / np.sqrt(n_subjects)
    
    # Get common time vectors (should be the same across subjects)
    feedback_times = all_results[0]['feedback_times']
    cue_times = all_results[0]['cue_times']
    
    # Aggregate cluster results if available
    cluster_results_available = any(result.get('cluster_results') is not None for result in all_results)
    aggregated_cluster_results = None
    
    if cluster_results_available:
        print("Aggregating cluster results across subjects...")
        aggregated_cluster_results = aggregate_cluster_results(all_results)
    
    aggregated_results = {
        'group_mean': group_mean,
        'group_std': group_std,
        'group_sem': group_sem,
        'individual_scores': all_scores,
        'feedback_times': feedback_times,
        'cue_times': cue_times,
        'n_subjects': n_subjects,
        'subject_ids': [result['subj_id'] for result in all_results],
        'cluster_results': aggregated_cluster_results
    }
    
    return aggregated_results

def aggregate_cluster_results(all_results):
    """
    Aggregate cluster results across subjects using MNE's implementation
    """
    # stack all the 'classification_scores' from all the results
    X = [all_results[x]['classification_scores'] for x in range(len(all_results))]
    X = np.stack(X)
    t_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(
        X, 
        n_permutations=1000,
        tail=0
        )
    
    print(f"MNE group cluster test completed")
    print(f"Found {len(clusters)} clusters")
    print(f"Significant clusters: {np.sum(cluster_p_values < 0.05)}")
    
    # Create aggregated cluster results
    aggregated_cluster_results = {
        't_obs': t_obs,
        'clusters': clusters,
        'cluster_p_values': cluster_p_values,
        'H0': H0,
    }
    
    return aggregated_cluster_results

def create_group_heatmap(aggregated_results, target_roi, significance_threshold=0.05):
    """
    Create group-level heatmap with statistical testing
    """
    if aggregated_results is None:
        print("No aggregated results to plot!")
        return
    
    group_mean = aggregated_results['group_mean']
    group_sem = aggregated_results['group_sem']
    feedback_times = aggregated_results['feedback_times']
    cue_times = aggregated_results['cue_times']
    n_subjects = aggregated_results['n_subjects']
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Main heatmap
    ax1 = axes[0, 0]
    im1 = ax1.imshow(group_mean, aspect='auto', origin='lower', 
                     cmap='RdBu_r', vmin=0.3, vmax=0.7,
                     extent=[cue_times[0], cue_times[-1], 
                            feedback_times[0], feedback_times[-1]])
    ax1.set_title(f'{target_roi} Reinstatement - Group Mean (n={n_subjects})')
    ax1.set_xlabel('Cue On Time (s)')
    ax1.set_ylabel('Feedback Start Time (s)')
    ax1.plot([cue_times[0], cue_times[-1]], [feedback_times[0], feedback_times[-1]], 
             'k--', alpha=0.5, linewidth=1)
    plt.colorbar(im1, ax=ax1, label='AUC Score')
    
    # SEM heatmap
    ax2 = axes[0, 1]
    im2 = ax2.imshow(group_sem, aspect='auto', origin='lower', 
                     cmap='viridis', vmin=0, vmax=np.percentile(group_sem, 95),
                     extent=[cue_times[0], cue_times[-1], 
                            feedback_times[0], feedback_times[-1]])
    ax2.set_title(f'{target_roi} Reinstatement - Standard Error')
    ax2.set_xlabel('Cue On Time (s)')
    ax2.set_ylabel('Feedback Start Time (s)')
    plt.colorbar(im2, ax=ax2, label='SEM')
    
    # Cluster-based significance (if available)
    ax3 = axes[1, 0]
    
    if aggregated_results.get('cluster_results') is not None:
        cluster_results = aggregated_results['cluster_results']
        
        # Create cluster mask
        cluster_mask = np.zeros_like(group_mean, dtype=bool)
        significant_clusters = []
        
        for i, (cluster, p_val) in enumerate(zip(cluster_results['clusters'], 
                                                cluster_results['cluster_p_values'])):
            if p_val < significance_threshold:
                cluster_mask[cluster] = True
                significant_clusters.append((cluster, p_val))
        
        # Plot original scores with significant clusters highlighted
        im3 = ax3.imshow(group_mean, aspect='auto', origin='lower', 
                         cmap='RdBu_r', vmin=0.3, vmax=0.7,
                         extent=[cue_times[0], cue_times[-1], 
                                feedback_times[0], feedback_times[-1]])
        
        # Overlay significant clusters
        if len(significant_clusters) > 0:
            # Create a mask for significant clusters only
            sig_cluster_mask = np.zeros_like(group_mean)
            for cluster, p_val in significant_clusters:
                sig_cluster_mask[cluster] = 1
            
            # Plot significant clusters with high contrast
            ax3.imshow(sig_cluster_mask, aspect='auto', origin='lower', 
                       cmap='Reds', alpha=0.7, vmin=0, vmax=1,
                       extent=[cue_times[0], cue_times[-1], 
                              feedback_times[0], feedback_times[-1]])
        
        ax3.set_title(f'{target_roi} Reinstatement - Significant Clusters (p<{significance_threshold})')
        
        # Cluster statistics summary
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        summary_text = f"Group Cluster Analysis Summary\\n\\n"
        summary_text += f"Subjects: {cluster_results['n_subjects']}\\n"
        summary_text += f"Total clusters found: {len(cluster_results['clusters'])}\\n"
        summary_text += f"Significant clusters: {len(significant_clusters)}\\n"
        
        if len(significant_clusters) > 0:
            summary_text += "Significant Clusters:\\n"
            for i, (cluster, p_val) in enumerate(significant_clusters):
                cluster_size = len(cluster[0])
                summary_text += f"  Cluster {i+1}: p={p_val:.4f}, size={cluster_size}\\n"
        
        ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, 
                 fontsize=10, verticalalignment='top', fontfamily='monospace')
        
    else:
        # Fallback to point-wise significance testing
        from scipy import stats
        
        # Perform t-test against chance (0.5)
        t_stats = np.zeros_like(group_mean)
        p_values = np.ones_like(group_mean)
        
        for i in range(group_mean.shape[0]):
            for j in range(group_mean.shape[1]):
                individual_scores = aggregated_results['individual_scores'][:, i, j]
                t_stat, p_val = stats.ttest_1samp(individual_scores, 0.5)
                t_stats[i, j] = t_stat
                p_values[i, j] = p_val
        
        # Create significance mask
        sig_mask = p_values < significance_threshold
        sig_plot = np.ma.masked_where(~sig_mask, group_mean)
        
        im3 = ax3.imshow(group_mean, aspect='auto', origin='lower', 
                         cmap='RdBu_r', vmin=0.3, vmax=0.7,
                         extent=[cue_times[0], cue_times[-1], 
                                feedback_times[0], feedback_times[-1]])
        ax3.imshow(sig_plot, aspect='auto', origin='lower', 
                   cmap='RdBu_r', vmin=0.3, vmax=0.7,
                   extent=[cue_times[0], cue_times[-1], 
                          feedback_times[0], feedback_times[-1]])
        ax3.set_title(f'{target_roi} Reinstatement - Significant Regions (p<{significance_threshold})')
        
        # Effect size (Cohen's d)
        ax4 = axes[1, 1]
        cohens_d = (group_mean - 0.5) / np.std(aggregated_results['individual_scores'], axis=0)
        im4 = ax4.imshow(cohens_d, aspect='auto', origin='lower', 
                         cmap='RdBu_r', vmin=-1, vmax=1,
                         extent=[cue_times[0], cue_times[-1], 
                                feedback_times[0], feedback_times[-1]])
        ax4.set_title(f'{target_roi} Reinstatement - Effect Size (Cohen\'s d)')
        plt.colorbar(im4, ax=ax4, label="Cohen's d")
    
    ax3.set_xlabel('Cue On Time (s)')
    ax3.set_ylabel('Feedback Start Time (s)')
    plt.colorbar(im3, ax=ax3, label='AUC Score')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n{target_roi} Group Analysis Summary:")
    print(f"  Subjects: {n_subjects}")
    print(f"  Mean AUC: {group_mean.mean():.3f} ± {group_mean.std():.3f}")
    print(f"  Max AUC: {group_mean.max():.3f}")
    
    if aggregated_results.get('cluster_results') is not None:
        cluster_results = aggregated_results['cluster_results']
        significant_clusters = [p_val for p_val in cluster_results['cluster_p_values'] if p_val < significance_threshold]
        print(f"  Significant clusters: {len(significant_clusters)}")
    else:
        sig_mask = p_values < significance_threshold
        print(f"  Significant timepoints: {np.sum(sig_mask)} / {sig_mask.size} ({100*np.sum(sig_mask)/sig_mask.size:.1f}%)")
    
    return fig

# Run aggregation and create group heatmap
if len(all_results) > 0:
    print(f"\nAggregating results from {len(all_results)} subjects...")
    aggregated_results = aggregate_subject_results(all_results)
    
    if aggregated_results is not None:
        print("Creating group-level heatmap...")
        fig = create_group_heatmap(aggregated_results, target_roi)
        
        # Save aggregated results
        aggregated_file = f'{roi_results_dir}/group_aggregated_results.npy'
        np.save(aggregated_file, aggregated_results)
        print(f"Aggregated results saved to: {aggregated_file}")
    else:
        print("Failed to aggregate results!")
else:
    print("No successful results to aggregate!")


In [ ]:
# stack all the 'classification_scores' from all the results
X = [all_results[x]['classification_scores'] for x in range(len(all_results))]
X = np.stack(X)
t_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(
    X, 
    n_permutations=1000,
    tail=0
    )

print(f"MNE group cluster test completed")
print(f"Found {len(clusters)} clusters")
print(f"Significant clusters: {np.sum(cluster_p_values < 0.05)}")

In [ ]:
np.mean([X[x, 150:250, :].mean() for x in range(len(X))])

In [ ]:
np.mean([X[x, 0:150, :].mean() for x in range(len(X))])

In [ ]:
tfr_data

In [ ]:
# Select one subject for analysis (you can change this)
selected_subj = subj_ids[-2]  # First subject
print(f"Selected subject: {selected_subj}")

# Define analysis parameters
day = 'Day1'
events = ['feedback_start', 'cue_on']
rois = ['OFC']  # Regions of interest:  'dmPFC', 'HPC', 'AMY', 'ACC

# TFR parameters
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

print(f"Frequency range: {freqs[0]:.1f} - {freqs[-1]:.1f} Hz")
print(f"Number of frequencies: {len(freqs)}")


In [ ]:
# Load electrode information
elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{selected_subj}/{day}_reref_elec_df')
print(f"Electrode info loaded for {selected_subj}")
print(f"Total electrodes: {len(elec_df)}")
print(f"Regions available: {elec_df.salman_region.unique()}")

# Filter for ROIs
roi_elecs = elec_df[elec_df.salman_region.isin(rois)]
print(f"Electrodes in ROIs: {len(roi_elecs)}")
for roi in rois:
    roi_count = len(roi_elecs[roi_elecs.salman_region == roi])
    print(f"  {roi}: {roi_count} electrodes")


## Load TFR Data for Both Events


In [ ]:
# Load TFR data for both events
tfr_data = {}
filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{selected_subj}/scratch/TFR'

for event in events:
    try:
        # Load TFR data
        tfr_file = f'{filepath}/{event}-tfr.h5'
        if os.path.exists(tfr_file):
            tfr_data[event] = mne.time_frequency.read_tfrs(tfr_file)
            print(f"Loaded {event} TFR: {tfr_data[event].data.shape}")
        else:
            print(f"TFR file not found: {tfr_file}")
            # Try alternative path structure
            alt_file = f'{filepath}/{event}/dfs/'
            if os.path.exists(alt_file):
                print(f"Found alternative path: {alt_file}")
                # We'll need to reconstruct TFR from individual electrode files
                tfr_data[event] = None
            else:
                print(f"No TFR data found for {event}")
    except Exception as e:
        print(f"Error loading {event}: {e}")
        tfr_data[event] = None


In [ ]:
# Check what we have loaded
for event in events:
    if tfr_data[event] is not None:
        print(f"\n{event} TFR info:")
        print(f"  Shape: {tfr_data[event].data.shape}")
        print(f"  Times: {tfr_data[event].times[0]:.3f} to {tfr_data[event].times[-1]:.3f} s")
        print(f"  Frequencies: {tfr_data[event].freqs[0]:.1f} to {tfr_data[event].freqs[-1]:.1f} Hz")
        print(f"  Channels: {len(tfr_data[event].ch_names)}")
    else:
        print(f"\n{event}: No TFR data available")


## Prepare Data for Classification


In [ ]:
# Load tfr_roi for the specific ROI
# This assumes 'roi_channels' is defined as a list of channel names for your ROI

tfr_roi = {}
roi_channels = roi_elecs.values
for event in events:
    if tfr_data[event] is not None:
        # Find indices of channels in ROI
        ch_indices = [i for i, ch in enumerate(tfr_data[event].ch_names) if ch in roi_channels]
        if not ch_indices:
            print(f"No ROI channels found in {event}")
            tfr_roi[event] = None
        else:
            # Pick only ROI channels
            tfr_roi[event] = tfr_data[event].copy().pick_channels([tfr_data[event].ch_names[i] for i in ch_indices])
            print(f"{event}: {len(ch_indices)} ROI channels loaded ({[tfr_data[event].ch_names[i] for i in ch_indices]})")
    else:
        tfr_roi[event] = None

tfr_roi 


10/21/2025: Basically, can we take RPE related signals during encoding [that predict neg, neutral, pos RPEs], amd test whether they generalize to the retrieval period. 

## Multi-Class RPE Classification Analysis


In [ ]:
if selected_subj is not None and all(tfr_roi[event] is not None for event in events):
    
    # Set up LDA classifier for multi-class classification
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    clf = make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis()
    )
    
    # Get data dimensions
    feedback_data = tfr_roi['feedback_start'].data  # shape: (trials, channels, freqs, times)
    cue_data = tfr_roi['cue_on'].data
    
    n_trials, n_channels, n_freqs, n_times_feedback = feedback_data.shape
    _, _, _, n_times_cue = cue_data.shape
    
    print(f"Data shapes:")
    print(f"  Feedback: {feedback_data.shape}")
    print(f"  Cue: {cue_data.shape}")
    print(f"  Labels: {reward_labels.shape}")
    
    # Get RPE categories from learn_df for the selected subject
    subj_learn_df = learn_df[learn_df.participant == selected_subj].copy()
    
    # Ensure we have the right number of trials
    if len(subj_learn_df) != n_trials:
        print(f"Warning: learn_df has {len(subj_learn_df)} trials, but TFR has {n_trials} trials")
        # Take the first n_trials from learn_df
        subj_learn_df = subj_learn_df.head(n_trials)
    
    # Extract RPE categories from learn_df
    if 'rpe_cat' in subj_learn_df.columns:
        rpe_cat = subj_learn_df['rpe_cat'].values
        print(f"\nUsing existing 'rpe_cat' column from learn_df")
    else:
        # Create RPE categories from RPE values if rpe_cat doesn't exist
        rpe_values = subj_learn_df['rpe'].values
        rpe_cat = []
        for rpe in rpe_values:
            if rpe < -0.1:
                rpe_cat.append('neg')
            elif rpe > 0.1:
                rpe_cat.append('pos')
            else:
                rpe_cat.append('neutral')
        rpe_cat = np.array(rpe_cat)
        print(f"\nCreated RPE categories from 'rpe' column")
    
    # Check distribution of RPE categories
    unique_cats, counts = np.unique(rpe_cat, return_counts=True)
    print(f"\nRPE category distribution:")
    for cat, count in zip(unique_cats, counts):
        print(f"  {cat}: {count} trials")
    
    # Ensure we have enough trials for each class
    min_trials_per_class = min(counts)
    if min_trials_per_class < 5:
        print(f"Warning: Some classes have < 5 trials. Min trials per class: {min_trials_per_class}")
    
    # Define HFA frequency band (70-200 Hz)
    freqs = tfr_roi['feedback_start'].freqs
    hfa_mask = (freqs >= 70) & (freqs <= 200)
    hfa_indices = np.where(hfa_mask)[0]
    
    print(f"HFA frequency range: {freqs[hfa_indices[0]]:.1f} - {freqs[hfa_indices[-1]]:.1f} Hz")
    print(f"HFA frequencies: {len(hfa_indices)} out of {n_freqs} total")
    
    # Average power across HFA frequencies
    feedback_hfa = np.mean(feedback_data[:, :, hfa_mask, :], axis=2)  # (trials, channels, times)
    cue_hfa = np.mean(cue_data[:, :, hfa_mask, :], axis=2)
    
    print(f"HFA averaged data shapes:")
    print(f"  Feedback HFA: {feedback_hfa.shape}")
    print(f"  Cue HFA: {cue_hfa.shape}")
    
    # Apply 200ms sliding window smoothing
    def apply_sliding_window(data, window_size_ms=200, sfreq=None):
        """
        Apply sliding window smoothing to reduce temporal resolution
        """
        if sfreq is None:
            # Estimate sampling rate from time vector
            times = tfr_roi['feedback_start'].times
            sfreq = 1 / (times[1] - times[0])
        
        window_size_samples = int(window_size_ms * sfreq / 1000)
        window_size_samples = max(1, window_size_samples)  # At least 1 sample
        
        print(f"Sliding window: {window_size_ms}ms = {window_size_samples} samples")
        
        smoothed_data = np.zeros_like(data)
        
        for trial in range(data.shape[0]):
            for channel in range(data.shape[1]):
                # Apply sliding window average
                for t in range(data.shape[2]):
                    start_idx = max(0, t - window_size_samples // 2)
                    end_idx = min(data.shape[2], t + window_size_samples // 2 + 1)
                    smoothed_data[trial, channel, t] = np.mean(data[trial, channel, start_idx:end_idx])
        
        return smoothed_data
    
    # Apply smoothing to both datasets
    print(f"\nApplying 200ms sliding window smoothing...")
    feedback_hfa_smooth = apply_sliding_window(feedback_hfa, window_size_ms=200)
    cue_hfa_smooth = apply_sliding_window(cue_hfa, window_size_ms=200)
    
    print(f"Smoothed data shapes:")
    print(f"  Feedback HFA smooth: {feedback_hfa_smooth.shape}")
    print(f"  Cue HFA smooth: {cue_hfa_smooth.shape}")
    
    # Initialize results matrix
    # Rows: feedback_start timepoints, Cols: cue_on timepoints
    classification_scores = np.zeros((n_times_feedback, n_times_cue))
    
    print(f"\nRunning multi-class LDA classification for {n_times_feedback} x {n_times_cue} = {n_times_feedback * n_times_cue} combinations...")
    
    # Define function for parallel classification
    def classify_timepoint_combination(args):
        fb_time_idx, cue_time_idx = args
        
        # Extract data for specific timepoints (HFA averaged, all electrodes, smoothed)
        X_train = feedback_hfa_smooth[:, :, fb_time_idx]  # (trials, channels)
        X_test = cue_hfa_smooth[:, :, cue_time_idx]
        
        # Train multi-class LDA classifier
        clf.fit(X_train, rpe_cat)
        
        # Test classifier - get prediction probabilities for all classes
        y_pred_proba = clf.predict_proba(X_test)  # (trials, n_classes)
        
        # Calculate multi-class AUC score
        # For multi-class, we can use macro-averaged AUC or one-vs-rest
        try:
            from sklearn.metrics import roc_auc_score
            from sklearn.preprocessing import LabelEncoder
            
            # Encode labels to numeric
            le = LabelEncoder()
            y_encoded = le.fit_transform(rpe_cat)
            
            # Calculate macro-averaged AUC (one-vs-rest)
            auc_scores = []
            for i in range(len(le.classes_)):
                y_binary = (y_encoded == i).astype(int)
                if len(np.unique(y_binary)) > 1:  # Check if class exists
                    auc = roc_auc_score(y_binary, y_pred_proba[:, i])
                    auc_scores.append(auc)
            
            if len(auc_scores) > 0:
                macro_auc = np.mean(auc_scores)
                return macro_auc
            else:
                return 0.5
                
        except ValueError:
            # Handle case where classification fails
            return 0.5
    
    # Create all combinations of timepoints (no frequency loop needed)
    combinations = []
    for fb_time_idx in range(n_times_feedback):
        for cue_time_idx in range(n_times_cue):
            combinations.append((fb_time_idx, cue_time_idx))
    
    print(f"Total combinations: {len(combinations)}")
    
    # Run parallel classification
    from joblib import Parallel, delayed
    
    # Use all available cores, but limit to avoid memory issues
    n_jobs = min(8, os.cpu_count())
    print(f"Using {n_jobs} parallel jobs...")
    
    results = Parallel(n_jobs=n_jobs, verbose=1)(
        delayed(classify_timepoint_combination)(combo) for combo in combinations
    )
    
    # Reshape results back to the classification_scores matrix
    result_idx = 0
    for fb_time_idx in range(n_times_feedback):
        for cue_time_idx in range(n_times_cue):
            classification_scores[fb_time_idx, cue_time_idx] = results[result_idx]
            result_idx += 1
    
    print("\nMulti-class LDA Classification completed!")
    print(f"Score range: {classification_scores.min():.3f} - {classification_scores.max():.3f}")
    
else:
    print("Cannot run classification - missing data")


## MVPA Classification Analysis


In [ ]:
# FIXED: Multi-Class RPE Classification Analysis
if selected_subj is not None and all(tfr_roi[event] is not None for event in events):
    
    # Set up LDA classifier for multi-class classification
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    clf = make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis()
    )
    
    # Get data dimensions
    feedback_data = tfr_roi['feedback_start'].data  # shape: (trials, channels, freqs, times)
    cue_data = tfr_roi['cue_on'].data
    
    n_trials, n_channels, n_freqs, n_times_feedback = feedback_data.shape
    _, _, _, n_times_cue = cue_data.shape
    
    print(f"Data shapes:")
    print(f"  Feedback: {feedback_data.shape}")
    print(f"  Cue: {cue_data.shape}")
    
    # Get RPE categories from learn_df for the selected subject
    subj_learn_df = learn_df[learn_df.participant == selected_subj].copy()
    
    # Ensure we have the right number of trials
    if len(subj_learn_df) != n_trials:
        print(f"Warning: learn_df has {len(subj_learn_df)} trials, but TFR has {n_trials} trials")
        subj_learn_df = subj_learn_df.head(n_trials)
    
    # Extract RPE categories from learn_df
    if 'rpe_cat' in subj_learn_df.columns:
        rpe_cat = subj_learn_df['rpe_cat'].values
        print(f"\nUsing existing 'rpe_cat' column from learn_df")
    else:
        # Create RPE categories from RPE values if rpe_cat doesn't exist
        rpe_values = subj_learn_df['rpe'].values
        rpe_cat = []
        for rpe in rpe_values:
            if rpe < -0.1:
                rpe_cat.append('neg')
            elif rpe > 0.1:
                rpe_cat.append('pos')
            else:
                rpe_cat.append('neutral')
        rpe_cat = np.array(rpe_cat)
        print(f"\nCreated RPE categories from 'rpe' column")
    
    # Check distribution of RPE categories
    unique_cats, counts = np.unique(rpe_cat, return_counts=True)
    print(f"\nRPE category distribution:")
    for cat, count in zip(unique_cats, counts):
        print(f"  {cat}: {count} trials")
    
    # Define HFA frequency band (70-200 Hz)
    freqs = tfr_roi['feedback_start'].freqs
    hfa_mask = (freqs >= 70) & (freqs <= 200)
    hfa_indices = np.where(hfa_mask)[0]
    
    print(f"HFA frequency range: {freqs[hfa_indices[0]]:.1f} - {freqs[hfa_indices[-1]]:.1f} Hz")
    print(f"HFA frequencies: {len(hfa_indices)} out of {n_freqs} total")
    
    # Average power across HFA frequencies
    feedback_hfa = np.mean(feedback_data[:, :, hfa_mask, :], axis=2)  # (trials, channels, times)
    cue_hfa = np.mean(cue_data[:, :, hfa_mask, :], axis=2)
    
    print(f"HFA averaged data shapes:")
    print(f"  Feedback HFA: {feedback_hfa.shape}")
    print(f"  Cue HFA: {cue_hfa.shape}")
    
    # Apply 200ms sliding window smoothing
    def apply_sliding_window(data, window_size_ms=200, sfreq=None):
        """
        Apply sliding window smoothing to reduce temporal resolution
        """
        if sfreq is None:
            # Estimate sampling rate from time vector
            times = tfr_data['feedback_start'].times
            sfreq = 1 / (times[1] - times[0])
        
        window_size_samples = int(window_size_ms * sfreq / 1000)
        window_size_samples = max(1, window_size_samples)  # At least 1 sample
        
        print(f"Sliding window: {window_size_ms}ms = {window_size_samples} samples")
        
        smoothed_data = np.zeros_like(data)
        
        for trial in range(data.shape[0]):
            for channel in range(data.shape[1]):
                # Apply sliding window average
                for t in range(data.shape[2]):
                    start_idx = max(0, t - window_size_samples // 2)
                    end_idx = min(data.shape[2], t + window_size_samples // 2 + 1)
                    smoothed_data[trial, channel, t] = np.mean(data[trial, channel, start_idx:end_idx])
        
        return smoothed_data
    
    # Apply smoothing to both datasets
    print(f"\nApplying 200ms sliding window smoothing...")
    feedback_hfa_smooth = apply_sliding_window(feedback_hfa, window_size_ms=200)
    cue_hfa_smooth = apply_sliding_window(cue_hfa, window_size_ms=200)
    
    print(f"Smoothed data shapes:")
    print(f"  Feedback HFA smooth: {feedback_hfa_smooth.shape}")
    print(f"  Cue HFA smooth: {cue_hfa_smooth.shape}")
    
    # Initialize results matrix
    # Rows: feedback_start timepoints, Cols: cue_on timepoints
    classification_scores = np.zeros((n_times_feedback, n_times_cue))
    
    print(f"\nRunning multi-class LDA classification for {n_times_feedback} x {n_times_cue} = {n_times_feedback * n_times_cue} combinations...")
    
    # Define function for parallel classification - FIXED VERSION
    def classify_timepoint_combination(args):
        fb_time_idx, cue_time_idx = args
        
        # Extract data for specific timepoints (HFA averaged, all electrodes, smoothed)
        X_train = feedback_hfa_smooth[:, :, fb_time_idx]  # (trials, channels)
        X_test = cue_hfa_smooth[:, :, cue_time_idx]
        
        # Train multi-class LDA classifier
        clf.fit(X_train, rpe_cat)
        
        # Test classifier - get prediction probabilities for all classes
        y_pred_proba = clf.predict_proba(X_test)  # (trials, n_classes)
        
        # Calculate multi-class AUC score using the SAME approach as debug
        try:
            from sklearn.metrics import roc_auc_score
            from sklearn.preprocessing import LabelEncoder
            
            # Encode labels to numeric (same as debug)
            le = LabelEncoder()
            y_encoded = le.fit_transform(rpe_cat)
            
            # Calculate macro-averaged AUC (one-vs-rest) - SAME as debug
            auc_scores = []
            for i in range(len(le.classes_)):
                y_binary = (y_encoded == i).astype(int)
                if len(np.unique(y_binary)) > 1:  # Check if class exists
                    auc = roc_auc_score(y_binary, y_pred_proba[:, i])
                    auc_scores.append(auc)
            
            if len(auc_scores) > 0:
                macro_auc = np.mean(auc_scores)
                return macro_auc
            else:
                return 0.5
                
        except ValueError:
            # Handle case where classification fails
            return 0.5
    
    # Create all combinations of timepoints
    combinations = []
    for fb_time_idx in range(n_times_feedback):
        for cue_time_idx in range(n_times_cue):
            combinations.append((fb_time_idx, cue_time_idx))
    
    print(f"Total combinations: {len(combinations)}")
    
    # Run parallel classification
    from joblib import Parallel, delayed
    
    # Use all available cores, but limit to avoid memory issues
    n_jobs = min(8, os.cpu_count())
    print(f"Using {n_jobs} parallel jobs...")
    
    results = Parallel(n_jobs=n_jobs, verbose=1)(
        delayed(classify_timepoint_combination)(combo) for combo in combinations
    )
    
    # Reshape results back to the classification_scores matrix
    result_idx = 0
    for fb_time_idx in range(n_times_feedback):
        for cue_time_idx in range(n_times_cue):
            classification_scores[fb_time_idx, cue_time_idx] = results[result_idx]
            result_idx += 1
    
    print("\nMulti-class LDA Classification completed!")
    print(f"Score range: {classification_scores.min():.3f} - {classification_scores.max():.3f}")
    print(f"Mean score: {classification_scores.mean():.3f}")
    print(f"Std score: {classification_scores.std():.3f}")
    
    # Check if we have meaningful variation
    if classification_scores.std() < 0.01:
        print("WARNING: Very low variation in scores - check if classification is working properly")
    else:
        print("Good variation in scores - classification appears to be working")
    
else:
    print("Cannot run classification - missing data")


## Create Reinstatement Heatmap


In [ ]:
if selected_subj is not None and 'classification_scores' in locals():
    
    # Create frequency-specific heatmaps for different frequency bands
    freq_bands = {
        'Theta': (2, 8),
        'Alpha': (8, 13),
        'Beta': (13, 30),
        'Gamma': (30, 100),
        'HFA': (70, 200)
    }
    
    freqs = tfr_roi['feedback_start'].freqs
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (band_name, (fmin, fmax)) in enumerate(freq_bands.items()):
        if i >= len(axes):
            break
            
        ax = axes[i]
        
        # Find frequency indices for this band
        freq_mask = (freqs >= fmin) & (freqs <= fmax)
        freq_indices = np.where(freq_mask)[0]
        
        if len(freq_indices) > 0:
            # Average across frequencies in this band
            band_scores = np.zeros((n_times_feedback, n_times_cue))
            
            for fb_time_idx in range(n_times_feedback):
                for cue_time_idx in range(n_times_cue):
                    # Average across frequencies in this band
                    band_scores[fb_time_idx, cue_time_idx] = classification_scores[fb_time_idx, cue_time_idx]
            
            # Create heatmap for this band
            im = ax.imshow(band_scores, 
                          aspect='auto', 
                          origin='lower',
                          cmap='RdBu_r',
                          vmin=0.3, vmax=0.7,
                          extent=[cue_times[0], cue_times[-1], feedback_times[0], feedback_times[-1]])
            
            ax.set_title(f'{band_name} ({fmin}-{fmax} Hz)')
            ax.set_xlabel('Cue On Time (s)')
            ax.set_ylabel('Feedback Start Time (s)')
            
            # Add diagonal line
            ax.plot([cue_times[0], cue_times[-1]], [feedback_times[0], feedback_times[-1]], 
                    'k--', alpha=0.5, linewidth=1)
            
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'No data\nfor {band_name}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{band_name} ({fmin}-{fmax} Hz)')
    
    # Remove empty subplot
    if len(freq_bands) < len(axes):
        fig.delaxes(axes[-1])
    
    plt.suptitle(f'Frequency-Specific Reinstatement Analysis - {selected_subj}', fontsize=16)
    plt.tight_layout()
    plt.show()
    
else:
    print("Cannot create frequency-specific analysis - missing data")
